# Train/Val/Test Splits

In [1]:
import sys; sys.path.append('..')
import json
import numpy as np
import pandas as pd
from src.data import load, split, feature_sets, prepare, temporal_split, log_target

## Regional split

In [2]:
df = load()
train, val, test = split(df, seed=42)

Checking how the split worked

In [3]:
for name, d in [("train", train), ("val", val), ("test", test)]:
    print(f"{name:5} {len(d):6} rows ({len(d)/len(df):.0%})  {d['REGION'].nunique()} regions",
          d["TARGET"].value_counts().sort_index().to_dict())

train  27973 rows (64%)  137 regions {'A': 17439, 'B': 8895, 'C': 1054, 'D-E': 378, 'F-G': 207}
val     7553 rows (17%)  30 regions {'A': 4630, 'B': 2447, 'C': 318, 'D-E': 106, 'F-G': 52}
test    8035 rows (18%)  30 regions {'A': 5210, 'B': 2336, 'C': 321, 'D-E': 107, 'F-G': 61}


The split is around 64, 17, 18 instead of 70, 15, 15 but this is because many of the regions differ in size from each other and move together. However, all of the sets have fires of all sizes.

In [4]:
features = feature_sets(df)["model3_all"]          # or model1/2/4
(X_train, y_train), (X_val, y_val), (X_test, y_test) = prepare(train, val, test, features)

Log-transform the acres so that the model can be used on the unevenly distributed fire sizes. 

In [5]:
y_train_log = log_target(train)
y_val_log   = log_target(val)
y_test_log  = log_target(test)

In [6]:
#checking that the standardization worked
print(X_train.shape, "train mean ≈ 0:", X_train.mean().round(3))

(27973, 86) train mean ≈ 0: -0.0


In [7]:
split_map = {**{r: "train" for r in train.REGION.unique()},
             **{r: "val"   for r in val.REGION.unique()},
             **{r: "test"  for r in test.REGION.unique()}}

with open("../docs/region_split_seed42.json", "w") as f:
    json.dump(split_map, f, indent=1, sort_keys=True)

## Temporal Split

In [8]:
df = load()
train_t, val_t, test_t = temporal_split(df)

In [9]:
for name, d in [("train", train_t), ("val", val_t), ("test", test_t)]:
    print(f"{name:5} {len(d):6} rows ({len(d)/len(df):.0%})",
          sorted(d["FIRE_YEAR"].unique()),
          d["TARGET"].value_counts().sort_index().to_dict())

train  26907 rows (62%) [np.int64(2016), np.int64(2017), np.int64(2018)] {'A': 17648, 'B': 7690, 'C': 1035, 'D-E': 346, 'F-G': 188}
val     6454 rows (15%) [np.int64(2019)] {'A': 4255, 'B': 1829, 'C': 257, 'D-E': 79, 'F-G': 34}
test   10200 rows (23%) [np.int64(2020)] {'A': 5376, 'B': 4159, 'C': 401, 'D-E': 166, 'F-G': 98}


In [10]:
(X_train_t, y_train_t), (X_val_t, y_val_t), (X_test_t, y_test_t) = prepare(train_t, val_t, test_t, features)

y_train_t_log = log_target(train_t)
y_val_t_log   = log_target(val_t)
y_test_t_log  = log_target(test_t)